# Data Pipeline
**Project:** COMM514 - Msc Research Project (Seafood Representation in British Print Media)   
**Author:** Pushpit  
**Supervisors:** Ruth Thurstan, Hywel Williams

## Pipeline Overview

1. Load all text files from Raw Data folder
2. Load all metadata Excel files
3. Join text + metadata on Gale Document Number
4. Clean & Deduplicate
5. Export `corpus_master.csv`

## 1. Imports and Configurations

In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ———————————CONFIG——————————————
RAW_DATA_PATH = Path(r"D:\University of Exeter\MSc Advanced Data Science Modules\COMM514 - MSc Research Project\Raw Data")
OUTPUT_PATH = Path(r"D:\University of Exeter\MSc Advanced Data Science Modules\COMM514 - Msc Research Project")

# Verify Path
print(f"Raw Data Path exists: {RAW_DATA_PATH.exists()}")
decades = [d.name for d in RAW_DATA_PATH.iterdir() if d.is_dir()]
print(f"Decades found: {sorted(decades)}")

Raw Data Path exists: True
Decades found: ['1920s', '1930s', '1940s', '1950s', '1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s']


## 2. Data Cleaning

### 2.1 Load All Text Files

In [2]:
def extract_gale_id(filename):
    """Extract Gale Document ID from filename e.g. _direct_Fish_Supplie_GR3223881019"""
    match = re.search(r'_([A-Z]+\d+)(?:\.txt)?$', filename) # _ followed by [A-Z] 1 or more uppercase letters (e.g. GR). \d+	One or more digits (e.g. 3223881019). ([A-Z]{2}\d+)	Capture the letters + digits as a group. (?:\.txt)?	Optionally match .txt. $ Must be at the end of the string
    return match.group(1) if match else None

text_records = list() # List of dictionaries. 1 dictionary per text file
skipped_files = list()

from zipfile import ZipFile

for zip_path in RAW_DATA_PATH.rglob("*.zip"): # rglob recursively searches directories. Each result is a path object
    # Extract year and decade from folder path
    year = zip_path.parent.name
    decade = zip_path.parent.parent.name
    print(f"Processing {decade}/{year}...")
    
    with ZipFile(zip_path) as z: # Read directly from the zipfiles without extracting
        for member in z.namelist():
            if not member.lower().endswith(".txt"):
                continue
            filename = Path(member).name
            gale_id = extract_gale_id(Path(member).stem) # stem returns file name without extension
            # Skip files without a Gale ID (e.g. README.txt)
            if gale_id is None:
                skipped_files.append(member)
                continue
            #Read the file contents
            try:
                text = z.read(member).decode("utf-8", errors="ignore").strip()
            except Exception as e:
                text = ""
                print(f"Error reading {filename}: {e}")
            text_records.append({
                'gale_id': gale_id,
                'year': year,
                'decade': decade,
                'filename': filename,
                'text': text,
                'text_len': len(text)
            })

# Convert to DataFrame
df_texts = pd.DataFrame(text_records)

print(f"Text files loaded: {len(df_texts):,}")
print(f"Missing Gale ID: {df_texts['gale_id'].isna().sum()}")
print(f"Empty text files: {(df_texts['text_len'] == 0).sum()}")
print(f"Skipped files: {len(skipped_files)}")

if skipped_files:
    print("\nSkipped Files")
    for skipped_file in skipped_files:
        print("- ",skipped_file)

Processing 1920s/1920...
Processing 1920s/1921...
Processing 1920s/1922...
Processing 1920s/1923...
Processing 1920s/1924...
Processing 1920s/1925...
Processing 1920s/1926...
Processing 1920s/1927...
Processing 1920s/1928...
Processing 1920s/1929...
Processing 1930s/1930...
Processing 1930s/1931...
Processing 1930s/1932...
Processing 1930s/1933...
Processing 1930s/1934...
Processing 1930s/1935...
Processing 1930s/1936...
Processing 1930s/1937...
Processing 1930s/1938...
Processing 1930s/1939...
Processing 1940s/1940...
Processing 1940s/1941...
Processing 1940s/1942...
Processing 1940s/1943...
Processing 1940s/1944...
Processing 1940s/1945...
Processing 1940s/1946...
Processing 1940s/1947...
Processing 1940s/1948...
Processing 1940s/1949...
Processing 1950s/1950...
Processing 1950s/1951...
Processing 1950s/1952...
Processing 1950s/1953...
Processing 1950s/1954...
Processing 1950s/1955...
Processing 1950s/1956...
Processing 1950s/1957...
Processing 1950s/1958...
Processing 1950s/1959...


In [3]:
df_texts.head()

,gale_id,year,decade,filename,text,text_len
0,GW3226072601,1920,1920s,'mail'_Mustard_&_Cre_GW3226072601.txt,'MAIL' MUSTARD & CRESSThe woman buys things li...,4834
1,EE1863375446,1920,1920s,'Military_Octopus'_EE1863375446.txt,gandard. •MILITARY OCTOPUS.'INDIAN PRESS CRITI...,668
2,GW3226065594,1920,1920s,'mother_Humber'_Memo_GW3226065594.txt,'MOTHER HUMBER' MEMORIAL FUND.SUBSCRIPTIONS. •...,2148
3,JA3234012449,1920,1920s,'phone_Disadvantages_JA3234012449.txt,'PHONE DISADVANTAGES.ACTION ABERDEEN FISH MERC...,1309
4,JA3234029148,1920,1920s,12_Months_in_Prison__JA3234029148.txt,12 MONTHS PRISON £1000 FINE.At Leeds Assizes S...,371


In [4]:
df_texts.shape

(298341, 6)

### 2.2 Load All Metadata CSV Files

In [5]:
metadata_frames = list() # One dataframe for each excel file. e.g df_1922, df_1926, ....

for csv_file in RAW_DATA_PATH.rglob("*metadata*.csv"): # Recursively search through the metadata files
    try:
        df = pd.read_csv(csv_file, index_col=False) # Document Title is being considered as index without index_col=False
        # Normalize Gale ID by stripping GALE| prefix
        if "Gale Document Number" in df.columns:
            df['gale_id'] = df['Gale Document Number'].astype(str).str.replace('GALE|', '', regex=False).str.strip()
        df['source_file'] = csv_file.name
        metadata_frames.append(df)
        print(f"Loaded {csv_file.name} ({len(df):,} rows)")
    except Exception as e:
        print(f"Error loading {csv_file.name}: {e}")

print(f"Metadata files found: {len(metadata_frames)}")

if not metadata_frames:
    raise ValueError("No metadata csv files found!")
    
df_metadata = pd.concat(metadata_frames, ignore_index=True)

print(f"\nTotal Metadata Rows: {len(df_metadata):,}")
print(f"Rows with no Gale ID: {df_metadata['gale_id'].isnull().sum()}")
print(f"Columns: {list(df_metadata.columns)}")

Loaded seafood_1920_metadata.csv (4,811 rows)
Loaded seafood_1921_metadata.csv (5,582 rows)
Loaded seafood_1922_metadata.csv (5,436 rows)
Loaded seafood_1923_metadata.csv (6,088 rows)
Loaded seafood_1924_metadata.csv (5,959 rows)
Loaded seafood_1925_metadata.csv (6,254 rows)
Loaded seafood_1926_metadata.csv (7,571 rows)
Loaded seafood_1927_metadata.csv (7,588 rows)
Loaded seafood_1928_metadata.csv (7,614 rows)
Loaded seafood_1929_metadata.csv (7,800 rows)
Loaded seafood_1930_metadata.csv (6,912 rows)
Loaded seafood_1931_metadata.csv (8,206 rows)
Loaded seafood_1932_metadata.csv (8,390 rows)
Loaded seafood_1933_metadata.csv (9,443 rows)
Loaded seafood_1934_metadata.csv (9,299 rows)
Loaded seafood_1935_metadata.csv (9,115 rows)
Loaded seafood_1936_metadata.csv (8,997 rows)
Loaded seafood_1937_metadata.csv (9,317 rows)
Loaded seafood_1938_metadata.csv (8,416 rows)
Loaded seafood_1939_metadata.csv (7,493 rows)
Loaded seafood_1940_metadata.csv (4,705 rows)
Loaded seafood_1941_metadata.csv (

In [6]:
df_metadata.head()

,Document Title,Content Type,Document Types,Publication Title,Publication Date,Publisher,Place of Publication,Author,Source Library,Gale Primary Source,Gale Document Number,Subject,Language,gale_id,source_file
0,Lyons Girl Striker Fined.,Newspaper,Article,The Times.,"August 27, 1920",NaN,"London, United Kingdom",NaN,Times Newspapers Limited,The Times Digital Archive,GALE|CS118427419,NaN,NaN,CS118427419,seafood_1920_metadata.csv
1,Local and District News,Newspaper,Article,Western Times.,"May 22, 1920",NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|GR3221018695,NaN,NaN,GR3221018695,seafood_1920_metadata.csv
2,Mother Humber memorial Fund,Newspaper,Article,Hull Daily Mail.,"April 15, 1920",NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|GW3226064605,NaN,NaN,GW3226064605,seafood_1920_metadata.csv
3,To-day's Markets,Newspaper,Article,Evening Telegraph.,"March 30, 1920",NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|JE3237043458,NaN,NaN,JE3237043458,seafood_1920_metadata.csv
4,Fish that Grow Faster,Newspaper,Article,Daily Mail.,"July 10, 1920",NaN,"London, United Kingdom",NaN,Associated Newspapers Limited,Daily Mail Historical Archive,GALE|EE1862965171,NaN,NaN,EE1862965171,seafood_1920_metadata.csv


In [7]:
# Check for duplicate Gale Ids in metadata files
print(f"Unique Gale IDs: {df_metadata['gale_id'].nunique():,}")
print(f"Duplicate Gale IDs: {df_metadata['gale_id'].duplicated().sum():,}")

Unique Gale IDs: 298,341
Duplicate Gale IDs: 0


In [8]:
# Check for duplicate Gale Ids in text files
print(f"Unique Gale IDs in text files: {df_texts['gale_id'].nunique():,}")
print(f"Duplicate Gale IDs: {df_texts['gale_id'].duplicated().sum():,}")

Unique Gale IDs in text files: 298,341
Duplicate Gale IDs: 0


### 2.3 Join Text and Metadata on Gale ID

In [9]:
df_corpus = df_texts.merge(df_metadata, on='gale_id', how='left') # keep every row from df_texts even if no metadata exists
matched = df_corpus['source_file'].notna().sum()
unmatched = df_corpus['source_file'].isna().sum()
print("Join Complete!")
print(f"Matched: {matched:,} ({matched / len(df_corpus)*100:.1f}%)")
print(f"Unmatched: {unmatched:,} ({unmatched / len(df_corpus)*100:.1f}%)")

Join Complete!
Matched: 298,341 (100.0%)
Unmatched: 0 (0.0%)


In [10]:
print(len(df_texts))
print(len(df_metadata))
print(len(df_corpus))

298341
298341
298341


In [11]:
df_corpus.head()

,gale_id,year,decade,filename,text,text_len,Document Title,Content Type,Document Types,Publication Title,Publication Date,Publisher,Place of Publication,Author,Source Library,Gale Primary Source,Gale Document Number,Subject,Language,source_file
0,GW3226072601,1920,1920s,'mail'_Mustard_&_Cre_GW3226072601.txt,'MAIL' MUSTARD & CRESSThe woman buys things li...,4834,'mail' Mustard & Cress,Newspaper,Article,Hull Daily Mail.,"August 31, 1920",NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|GW3226072601,NaN,NaN,seafood_1920_metadata.csv
1,EE1863375446,1920,1920s,'Military_Octopus'_EE1863375446.txt,gandard. •MILITARY OCTOPUS.'INDIAN PRESS CRITI...,668,'Military Octopus',Newspaper,Article,Daily Mail.,"October 2, 1920",NaN,"London, United Kingdom",From our Own Correspondent,Associated Newspapers Limited,Daily Mail Historical Archive,GALE|EE1863375446,NaN,NaN,seafood_1920_metadata.csv
2,GW3226065594,1920,1920s,'mother_Humber'_Memo_GW3226065594.txt,'MOTHER HUMBER' MEMORIAL FUND.SUBSCRIPTIONS. •...,2148,'mother Humber' Memorial Fund,Newspaper,Article,Hull Daily Mail.,"April 28, 1920",NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|GW3226065594,NaN,NaN,seafood_1920_metadata.csv
3,JA3234012449,1920,1920s,'phone_Disadvantages_JA3234012449.txt,'PHONE DISADVANTAGES.ACTION ABERDEEN FISH MERC...,1309,'phone Disadvantages,Newspaper,Article,Aberdeen Journal.,"February 14, 1920",NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|JA3234012449,NaN,NaN,seafood_1920_metadata.csv
4,JA3234029148,1920,1920s,12_Months_in_Prison__JA3234029148.txt,12 MONTHS PRISON £1000 FINE.At Leeds Assizes S...,371,12 Months in Prison and a £1000 Fine,Newspaper,Article,Aberdeen Journal.,"July 26, 1920",NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|JA3234029148,NaN,NaN,seafood_1920_metadata.csv


### 2.4 Data Cleaning

In [12]:
df_corpus = df_corpus[df_corpus['text_len'] > 0] # Remove empty text files
print(f"After removing empty text files: {len(df_corpus):,}")

After removing empty text files: 298,331


In [13]:
df_corpus['year'] = df_corpus['year'].astype('Int64') # Standardize year column as integer

In [14]:
# Standarize Publication Date
df_corpus['Publication Date'] = df_corpus['Publication Date'].replace('[April 5, 2020]', 'April 5, 2020') # 1 row with incorrect format
df_corpus['Publication Date Timestamp'] = pd.to_datetime(df_corpus['Publication Date'], errors='coerce')

In [15]:
df_corpus['Publication Date']

0           August 31, 1920
1           October 2, 1920
2            April 28, 1920
3         February 14, 1920
4             July 26, 1920
                ...        
298336      October 4, 2024
298337     October 16, 2024
298338       April 20, 2024
298339        April 6, 2024
298340     January 20, 2024
Name: Publication Date, Length: 298331, dtype: str

In [16]:
df_corpus['Publication Date Timestamp']

0        1920-08-31
1        1920-10-02
2        1920-04-28
3        1920-02-14
4        1920-07-26
            ...    
298336   2024-10-04
298337   2024-10-16
298338   2024-04-20
298339   2024-04-06
298340   2024-01-20
Name: Publication Date Timestamp, Length: 298331, dtype: datetime64[us]

In [17]:
# Clean Publication Title
df_corpus['Publication Title'] = df_corpus['Publication Title'].str.strip().str.replace(r'\.$', '', regex=True)

### 2.5 Export Master Corpus

In [18]:
output_file = OUTPUT_PATH / "corpus_master_final.pkl"
df_corpus.to_pickle(output_file)
print(f"Saved Master Corpus!")

Saved Master Corpus!


In [19]:
df_corpus = pd.read_pickle(OUTPUT_PATH / "corpus_master_final.pkl")
df_corpus.head()

,gale_id,year,decade,filename,text,text_len,Document Title,Content Type,Document Types,Publication Title,...,Publisher,Place of Publication,Author,Source Library,Gale Primary Source,Gale Document Number,Subject,Language,source_file,Publication Date Timestamp
0,GW3226072601,1920,1920s,'mail'_Mustard_&_Cre_GW3226072601.txt,'MAIL' MUSTARD & CRESSThe woman buys things li...,4834,'mail' Mustard & Cress,Newspaper,Article,Hull Daily Mail,...,NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|GW3226072601,NaN,NaN,seafood_1920_metadata.csv,1920-08-31
1,EE1863375446,1920,1920s,'Military_Octopus'_EE1863375446.txt,gandard. •MILITARY OCTOPUS.'INDIAN PRESS CRITI...,668,'Military Octopus',Newspaper,Article,Daily Mail,...,NaN,"London, United Kingdom",From our Own Correspondent,Associated Newspapers Limited,Daily Mail Historical Archive,GALE|EE1863375446,NaN,NaN,seafood_1920_metadata.csv,1920-10-02
2,GW3226065594,1920,1920s,'mother_Humber'_Memo_GW3226065594.txt,'MOTHER HUMBER' MEMORIAL FUND.SUBSCRIPTIONS. •...,2148,'mother Humber' Memorial Fund,Newspaper,Article,Hull Daily Mail,...,NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|GW3226065594,NaN,NaN,seafood_1920_metadata.csv,1920-04-28
3,JA3234012449,1920,1920s,'phone_Disadvantages_JA3234012449.txt,'PHONE DISADVANTAGES.ACTION ABERDEEN FISH MERC...,1309,'phone Disadvantages,Newspaper,Article,Aberdeen Journal,...,NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|JA3234012449,NaN,NaN,seafood_1920_metadata.csv,1920-02-14
4,JA3234029148,1920,1920s,12_Months_in_Prison__JA3234029148.txt,12 MONTHS PRISON £1000 FINE.At Leeds Assizes S...,371,12 Months in Prison and a £1000 Fine,Newspaper,Article,Aberdeen Journal,...,NaN,"London, United Kingdom",NaN,British Library,British Library Newspapers,GALE|JA3234029148,NaN,NaN,seafood_1920_metadata.csv,1920-07-26


In [20]:
print(len(df_corpus))

298331
